In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
from tqdm import tqdm    # Shows progress bar
import torch.optim as optim
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
import kagglehub
import os
import random
import numpy as np
import torchdata
import torch.nn as nn

import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch.optim as optim

from sklearn.metrics import confusion_matrix

import kagglehub
import os
import glob

import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

from PIL import Image
from torchvision.datasets import ImageFolder

from sklearn.model_selection import train_test_split



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here

# 3) Use RandomRotation(15) Augmentation on the training dataset + Reize the images to 32x32
# Define transformations / Augmentations

transform = transforms.Compose([
    transforms.Resize((32, 32)),                                              # Resize images
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

# No Augmentations on Validation & Testing ; Apply basic transformations to prepare the images
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),                                              # Resize images
    transforms.ToTensor(),
])

In [ ]:
# 1) Create a custom Dataset class or use ImageFolder (if applicable)
# 2) Create the training and testing datasets, and their DataLoaders

# First Get Paths Each train, test
train_dir = os.path.join(path, "PlantVillage", "train")
test_dir = os.path.join(path, "PlantVillage", "test")


# Image Folder
# Automatically handles everything if folders are named correctly
train_dataset = ImageFolder(
    root=train_dir,
    transform=transform
    )

test_dataset  = ImageFolder(
    root=test_dir,
    transform=test_transform
    )

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# DataLoader
batch_size =32


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,                       # Groups data into batches of BS
    shuffle=True,                                # for Training (prevents learning order patterns)
    num_workers=2
    )


valid_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,                                # No shuffle on Test
    num_workers=2
    )


# Get one batch of training images
images, labels = next(iter(train_loader))                                       # randomly selects 32 indices - calls dataset[idx] 32 times - stacks images → tensor - stacks labels → tensor

print(f"Batch shape: {images.shape}, \nLabels: {labels}")

In [ ]:
# 4) Display some sample images with their labels
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
from torchvision.datasets import ImageFolder
from tqdm import tqdm


# Get a batch of training data
data_iter = iter(train_loader)
images, labels = next(data_iter)

# CIFAR-10 class names
classes = ["Early_blight",
            "Late_blight",
            "healthy"]

# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = np.transpose(img.numpy(), (1, 2, 0))  # Convert (C, H, W) to (H, W, C)

    ax.imshow(img)
    ax.set_title(classes[labels[i].item()])
    ax.axis("off")

plt.show()

print("Shape of one image tensor:", images[0].shape)  # Expected: (3, 32, 32)

In [ ]:
# Output Size = (( N + 2P - F ) / S ) + 1                    - Floor it

# Input: [3, 32, 32]


# Conv 1 Input:   (128, 3, 32, 32)
print("\nAfter Conv:  ", (( 32 + 2 - 3 ) / 1 ) + 1 )

# Pool 1 Input:   (64, 128, 32, 32)
print("\nAfter Pool:  ",  32/2 )


# Conv 2 Input:   (64, 64, 16, 16)
print("\nAfter Conv:  ", (( 16 + 2 - 3 ) / 1 ) + 1 )

# Pool 2 Input:   (60, 64, 16, 12)
print("\nAfter Pool:  ",  16/2 )


# Conv 3 Input:   (64, 64, 8, 8)
print("\nAfter Conv:  ", (( 8 + 2 - 3 ) / 1 ) + 1 )

# Pool 3 Input:   (60, 64, 16, 12)
print("\nAfter Pool:  ",  8/2 )


# Conv 4 Input:   (64, 64, 4, 4)
print("\nAfter Conv:  ", (( 4 + 2 - 3 ) / 1 ) + 1 )


# Conv 4 Input:   (64, 64, 4, 4)
print("\nAfter Conv:  ", (( 4 + 2 - 3 ) / 1 ) + 1 )



In [ ]:
# Write your code here
# Define CNN Model

class CNNModel(nn.Module):
    def __init__(self, num_classes=3):
        """
        1️⃣ Define all layers in the model.
        """
        super(CNNModel, self).__init__()                                     # Constructor method - Used to define all layers once when the model is created


        self.features = nn.Sequential(                                          # “I will define the convolution/pooling feature extractor as a pipeline"

            # 1. Convolutional Layer                                                ( C, Feture Map, K, P )
            nn.Conv2d(1, 64, kernel_size=3, padding=1),                         #
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                                                    # halves H and W:


            # 2. Convolutional Layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),                        # [B,64,14,14] ( C, Feture Map, K, P )
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                                                    # halves H and W:  [128,64,14,14] -> [B,64,7,7]


            # 3. Convolutional Layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),                        # ( C, Feture Map, K, P )
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 4. Convolutional Layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),                        # ( C, Feture Map, K, P )
            nn.BatchNorm2d(128),
            nn.ReLU(),


            # 5. Convolutional Layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),                        #  ( C, Feture Map, K, P )
            nn.BatchNorm2d(128),
            nn.ReLU()

        )

        # ----------------------------------------------------------------------

        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),

            # Calculate input features (channels × height × width)
            # After 5 MaxPool2d(2):
            nn.Linear( 7 * 7 * 64 , 128),

            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

        # NO  Softmax Layer



    def forward(self, x):
        """
        2️⃣ Define the forward pass (how data flows through the model).
        """

        # Pass x through features                                        ; applies conv→relu→pool→conv→relu→pool automatically
        x = self.features(x)

        #  Pass result through classifier                                 ; flattens + linear layers
        x = self.classifier(x)

        return x


In [ ]:
# Write your code here
'''
Fun takes:
  -  logits: raw outputs from the model,    shape [B, num_classes]
  -  labels: true class indices,            shape [B]

'''

def accuracy_from_logits(logits, labels):
    # Get predicted class indices
    #Use torch.argmax with dim=1
    preds = torch.argmax(logits, dim=1)                                         # picks the index of the largest logit per sample ; the best class for each row in [B, num_classes].

    # Calculate and return accuracy
    # Compare preds with labels, convert to float, mean, then .item()
    return (preds == labels).float().mean().item()                              # 1) gives a boolean tensor like [True, False, True...] 2) convert booleans → float (True=1.0, False=0.0) 3) take mean → accuracy 4) .item() → return Python number




'''
Fun takes:
  -  model
  -  loader:
  -  optimizer:
  -  criterion:
'''

def train_one_epoch(model, loader, optimizer, criterion):

    # Sets training mode                                                        ; enables Dropout randomness  +  BatchNorm updates running stats (if used)
    model.train()

    # initilaize
    total_loss, total_acc = 0.0, 0.0


    # Loop over batches from DataLoader with a progress bar
    for batch in tqdm(loader):

        # Get images + labels [B]
        images, labels = batch['image'], batch['label']                         # Because our dataset returns a dict, each batch is a dict too

        #Move data to same device as model (CPU/GPU)
        images, labels = images.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        #-----------------------------------------------------------Forward pass
        logits = model(images)

        # Compute loss- CrossEntropyLoss expects logits + class indices
        loss = criterion(logits, labels)


        #----------------------------------------------------------Backward pass
        # Backward pass
        loss.backward()

        #Update parameters
        optimizer.step()

        # ------------------------------------------------------------Track Loss
        total_loss += loss.item()

        # --------------------------------------------------------Track Accuracy
        # detach(): stops tracking gradients for accuracy calc (not needed)
        total_acc += accuracy_from_logits(logits.detach(), labels)


    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
def evaluate(model, loader, criterion):

    # Set Model to Evaluation Mode                                              ;  Dropout → disabled  -   BatchNorm → uses stored statistics
    model.eval()

    # Initiliaze                                                                ; Tracking average loss and accuracy across all batches
    total_loss, total_acc = 0.0, 0.0


    # Disable Gradient calculation                                              ; No Backpropagation in Validation
    with torch.no_grad():

        for batch in tqdm(loader):

            # Get images [B, 1, 28, 28] + labels [B]
            images, labels = batch['image'], batch['label']                         # Because our dataset returns a dict, each batch is a dict too

            #Move data to same device as model (CPU/GPU)
            images, labels = images.to(device), labels.to(device)


            #Get model predictions
            logits = model(images)

            # Calculate loss
            loss = criterion(logits, labels)

            total_loss += loss.item()

            # Notice here we don’t need .detach() because gradients are already off.
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here
# Chooses GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Create instance of your CNN model  +  Move it to a device (CPU or GPU)
model = CNNModel().to(device)


# Debug
# let's have a look at our model architecture
print(model)
print("\n\n", next(model.parameters()).device)



# Training setup
# What loss function works for multi-class classification?
criterion = nn.CrossEntropyLoss()


# learning rate
learning_rate = 0.001


# optimizer
optimizer = optim.Adam(model.parameters(), lr= learning_rate)


#number of epochs
num_epochs = 10



# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}


# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
        )
    test_loss, test_acc = evaluate(
        model,
        valid_loader,
        criterion
        )

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
# Write your code here
